In [13]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta
import plotly.graph_objects as go
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
import src.bayesian_engine as be


In [14]:
theta = np.linspace(0,1,1000)
alpha_prior = 1
beta_prior = 1
prior_pdf = beta.pdf(theta,alpha_prior,beta_prior)


In [15]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=theta,
        y=prior_pdf,
        mode="lines",
        name="Prior Beta(1,1)",
    )
)

fig.update_layout(
    title="Prior Distribution",
    xaxis_title="Conversion Rate (θ)",
    yaxis_title="Density",
)

fig.show()

In [16]:
import src.bayesian_engine as be

print(be.__file__)
print(dir(be))

d:\projects\AB testing\src\bayesian_engine.py
['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'posterior_mean', 'posterior_varinace', 'update_posterior']


In [17]:
visitors = 10
conversions = 2

alpha_post, beta_post = be.update_posterior(
    alpha_prior=alpha_prior,
    beta_prior=beta_prior,
    conversion=conversions,
    visitors=visitors,
)
posterior_pdf = beta.pdf(theta, alpha_post, beta_post)

In [18]:
fig.add_trace(
    go.Scatter(
        x=theta,
        y=posterior_pdf,
        mode="lines",
        name=f"Posterior Beta({alpha_post}, {beta_post})",
    )
)

fig.update_layout(
    title="Prior vs Posterior Distribution"
)

fig.show()

In [19]:
scenarios = [
    (10, 2),
    (50, 10),
    (200, 40),
    (1000, 200),
]

In [20]:
for visitors, conversions in scenarios:

    alpha_post, beta_post = be.update_posterior(
        alpha_prior,
        beta_prior,
        conversions,
        visitors,
    )

    posterior_pdf = beta.pdf(theta, alpha_post, beta_post)

    fig.add_trace(
        go.Scatter(
            x=theta,
            y=posterior_pdf,
            mode="lines",
            name=f"{visitors} Visitors",
        )
    )
    fig.update_layout(
    title="Posterior Evolution with Increasing Sample Size",
    xaxis_title="Conversion Rate (θ)",
    yaxis_title="Density",
)

fig.show()

In [ ]:

frames = []

for visitors, conversions in scenarios:

    alpha_post, beta_post = be.update_posterior(
        alpha_prior,
        beta_prior,
        conversions,
        visitors,
    )

    posterior_pdf = beta.pdf(theta, alpha_post, beta_post)

    frame_name = "Prior" if visitors == 0 else f"{visitors} Visitors"

    frames.append(
        go.Frame(
            data=[
                go.Scatter(
                    x=theta,
                    y=posterior_pdf,
                    mode="lines",
                    name="Posterior",
                )
            ],
            name=frame_name,
        )
    )


fig = go.Figure(
    data=[
        go.Scatter(
            x=theta,
            y=frames[0].data[0].y,
            mode="lines",
            name="Posterior",
        )
    ],
    frames=frames,
)


fig.update_layout(
    title="Bayesian Posterior Updating",
    xaxis_title="Conversion Rate (θ)",
    yaxis_title="Density",
    xaxis=dict(range=[0, 1]),
    yaxis=dict(range=[0, max(prior_pdf) * 4]),
    template="plotly_white",
    updatemenus=[
        dict(
            type="buttons",
            showactive=False,
            buttons=[
                dict(
                    label="▶ Play",
                    method="animate",
                    args=[
                        None,
                        dict(
                            frame=dict(duration=1000, redraw=True),
                            transition=dict(duration=500),
                            fromcurrent=True,
                        ),
                    ],
                )
            ],
        )
    ],
)

fig.show()